# Project #07 — ETL Progress Logs
**Raúl Cetina · UPY · Unit 2 · Commute & Well-Being**

---

## Pregunta de investigación
> *Is sun exposure during the commute linked to academic stress, burnout or sleep quality?*
> (¿La exposición al sol durante el traslado se relaciona con el estrés académico, el burnout o la calidad del sueño?)

**Enfoque de este avance (proxy conductual):** la exposición al sol/calor durante el traslado deja una huella medible en la plataforma compartida: **compras de hidratación** (agua y electrolitos) justo después de llegar al campus. Cruzo esas compras con el **clima real de Mérida** (temperatura e índice UV) para construir los primeros indicadores del fenómeno. Las variables de estrés/sueño se integrarán después vía encuesta.

## Fuentes conectadas (3)
| # | Fuente | Tecnología | Qué aporta |
|---|--------|-----------|------------|
| 1 | **PostgreSQL de la plataforma compartida** (Ágora Campus / UPY-Ecommerce, Neon) | `psycopg2` + SQL | Compras en vivo: órdenes, artículos, productos |
| 2 | **CSV históricos del repo compartido** (GitHub raw) | `pandas.read_csv` | Catálogo y órdenes de muestra del equipo |
| 3 | **API Open-Meteo** (Mérida, sin API key) | `requests` | Temperatura máx. e índice UV diarios |

## Avance completado en este checkpoint
- ✔ Conexión a **2+ fuentes** de la plataforma compartida (con logs visibles)
- ✔ **Poller** que inserta compras simuladas y extracción por **polling** (tiempo real simulado)
- ✔ **Limpieza**: zona horaria, duplicados, nulos, tipos y normalización de texto
- ✔ **Indicador** ligado a la pregunta: *Índice de Hidratación Matutina post-traslado (IHM)* + correlación diaria compras↔UV/temperatura
- ✔ **ETL visual**: diagrama del pipeline y gráficas del indicador

*Checkpoint, no producto final: al final se listan limitaciones y próximos pasos.*

## 0 · Configuración

In [ ]:
%pip install -q psycopg2-binary pandas matplotlib requests

import warnings
import psycopg2
import pandas as pd
import requests
import matplotlib.pyplot as plt
import unicodedata
import random, time, uuid
from datetime import datetime, timezone

# pandas avisa que psycopg2 no es SQLAlchemy; para este avance es suficiente.
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")

# --- Plataforma compartida (Neon Postgres, endpoint pooler) ---
DB_URL = (
    "postgresql://neondb_owner:npg_Fr2iLGS8tZHm"
    "@ep-delicate-hat-atmk8wg6-pooler.c-9.us-east-1.aws.neon.tech/neondb"
    "?sslmode=require"
)
# --- CSVs del repositorio compartido del equipo ---
RAW = "https://raw.githubusercontent.com/DaniWin02/UPY-Ecommerce/main/data/samples/csv/"
# --- Mérida, Yucatán (para Open-Meteo) ---
LAT, LON = 20.97, -89.62
# Color institucional UPY para todas las gráficas (una sola serie)
MORADO, DORADO = "#4B2385", "#D9A33C"

print("Setup OK ·", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))
print("pandas", pd.__version__, "| psycopg2", psycopg2.__version__)

## 1 · EXTRACT — Fuente 1: PostgreSQL de la plataforma (en vivo)
Conexión directa a la base de datos del marketplace del equipo. Log de conexión + conteos + vista previa de las compras de hidratación.

In [ ]:
conn = psycopg2.connect(DB_URL)
cur = conn.cursor()

cur.execute("select version()")
print("[conexión 1 · OK] ", cur.fetchone()[0].split(" on ")[0])

for tabla in ("orders", "order_items", "products", "users"):
    cur.execute(f"select count(*) from {tabla}")
    print(f"    {tabla:<12} -> {cur.fetchone()[0]:>5} filas")

SQL_HIDRATACION = """
    select o.id as order_id, o.created_at, o.estado, oi.cantidad,
           oi.precio_unit, p.nombre as producto, o.referencia_pago
    from order_items oi
    join orders   o on o.id = oi.order_id
    join product_variants pv on pv.id = oi.variant_id
    join products p on p.id = pv.product_id
    where p.nombre ~* 'agua|electrolit|hidrat|suero'
      and o.estado in ('pago_verificado','preparando','listo_entrega','entregado')
    order by o.created_at
"""
df_db = pd.read_sql(SQL_HIDRATACION, conn)
print(f"\n[extract 1] compras de hidratación en la plataforma: {len(df_db)} filas")
df_db.tail(5)

## 2 · EXTRACT — Fuente 2: CSVs históricos del repo compartido
Segunda fuente de la plataforma: los datos de muestra publicados en el repositorio del equipo (GitHub raw).

In [ ]:
df_prod_hist = pd.read_csv(RAW + "products.csv")
df_ord_hist  = pd.read_csv(RAW + "orders.csv")
print(f"[conexión 2 · OK] products.csv -> {df_prod_hist.shape[0]} filas x {df_prod_hist.shape[1]} cols")
print(f"[conexión 2 · OK] orders.csv   -> {df_ord_hist.shape[0]} filas x {df_ord_hist.shape[1]} cols")
print("\nCatálogo histórico (muestra):")
df_prod_hist.head(3)

## 3 · EXTRACT — Fuente 3: clima real de Mérida (Open-Meteo)
Temperatura máxima e índice UV de los últimos 14 días — la variable de **exposición al sol** de mi pregunta.

In [ ]:
resp = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": LAT, "longitude": LON,
        "daily": "temperature_2m_max,uv_index_max",
        "timezone": "America/Merida", "past_days": 14, "forecast_days": 1,
    },
    timeout=15,
)
print("[conexión 3 · OK] Open-Meteo HTTP", resp.status_code)
crudo = resp.json()["daily"]
df_clima = pd.DataFrame({
    "fecha": pd.to_datetime(crudo["time"]).date,
    "temp_max": crudo["temperature_2m_max"],
    "uv_max": crudo["uv_index_max"],
})
print(f"[extract 3] {len(df_clima)} días de clima para Mérida")
df_clima.tail(5)

## 4 · Tiempo real simulado: POLLER + extracción con POLLING
Un **poller** inserta compras simuladas en la plataforma (probabilidad ponderada por la hora local y el UV real de hoy) y, en el mismo ciclo, el extractor hace **polling** leyendo solo las filas nuevas desde la última marca de tiempo — el patrón de un ETL incremental en tiempo real.

*Las compras simuladas son identificables: comprador `etl.simulador@upy.edu.mx`, referencia `SIM-…`, SKUs `ETL-AGUA-600` / `ETL-ELECTRO-625`.*

In [ ]:
# Referencias fijas del simulador (se resuelven una sola vez)
cur.execute("select id from users where email = 'etl.simulador@upy.edu.mx'")
COMPRADOR_ID = cur.fetchone()[0]
VARIANTES = {}
for sku in ("ETL-AGUA-600", "ETL-ELECTRO-625"):
    cur.execute("""select pv.id, coalesce(pv.precio_comunidad, pv.precio), p.vendor_id
                   from product_variants pv join products p on p.id = pv.product_id
                   where pv.sku = %s""", (sku,))
    vid, precio, vendor_id = cur.fetchone()
    VARIANTES[sku] = (vid, float(precio), vendor_id)

UV_HOY = float(df_clima.iloc[-1]["uv_max"])
FACTOR_UV = min(UV_HOY / 8.0, 1.5)
print(f"[poller] UV de hoy: {UV_HOY:.1f} -> factor de sed {FACTOR_UV:.2f}")

PESO_HORA = {7:.9, 8:1, 9:.8, 10:.5, 11:.4, 12:.6, 13:.9, 14:.8, 15:.6, 16:.4, 17:.3, 18:.3}

def ahora_db():
    """Marca de tiempo del RELOJ DE LA BD.
    Detalle fino de Postgres: now() se CONGELA al inicio de la transacción,
    así que usamos clock_timestamp() (hora real) y cerramos la transacción
    con commit para que el siguiente INSERT arranque una transacción nueva
    (su created_at = now() sería el inicio de ESTA si no la cerráramos)."""
    cur.execute("select clock_timestamp()")
    marca = cur.fetchone()[0]
    conn.commit()
    return marca

def insertar_compra():
    sku = random.choices(list(VARIANTES), weights=[0.7, 0.3])[0]
    vid, precio, vendor_id = VARIANTES[sku]
    qty = random.choice([1, 1, 1, 2])
    ref = f"SIM-{uuid.uuid4().hex[:8].upper()}"
    cur.execute("""insert into orders (comprador_id, vendor_id, estado, total,
                     referencia_pago, metodo_entrega, aula)
                   values (%s,%s,'entregado',%s,%s,'punto','Cafetería UPY') returning id""",
                (COMPRADOR_ID, vendor_id, precio*qty, ref))
    oid = cur.fetchone()[0]
    cur.execute("insert into order_items (order_id, variant_id, cantidad, precio_unit) values (%s,%s,%s,%s)",
                (oid, vid, qty, precio))
    cur.execute("""insert into payments (order_id, metodo, referencia, monto_declarado, estado, verificado_en)
                   values (%s,'efectivo',%s,%s,'verificado', now())""", (oid, ref, precio*qty))
    conn.commit()
    return f"{qty}x {sku} (${precio*qty:.2f})"

# --- ciclo poller + polling (marcas de tiempo SIEMPRE del reloj de la BD) ---
ultima_marca = ahora_db()
acumuladas = 0
for ciclo in range(1, 6):
    hora_local = (datetime.now(timezone.utc).hour - 6) % 24
    # prob mínima 50% en la demo para que se VEAN capturas en vivo
    prob = max(min(PESO_HORA.get(hora_local, 0.2) * FACTOR_UV, 0.95), 0.5)
    if random.random() < prob:
        print(f"  [poller  {ciclo}] inserta -> {insertar_compra()}")
    else:
        print(f"  [poller  {ciclo}] sin compra esta ronda (prob {prob:.0%})")
    time.sleep(2)
    nuevas = pd.read_sql(
        "select o.created_at, p.nombre, oi.cantidad from order_items oi "
        "join orders o on o.id = oi.order_id "
        "join product_variants pv on pv.id = oi.variant_id "
        "join products p on p.id = pv.product_id "
        "where o.created_at > %(ts)s and p.nombre ~* 'agua|electrolit'",
        conn, params={"ts": ultima_marca})
    conn.commit()  # cierra la transacción de lectura (misma razón de arriba)
    ultima_marca = ahora_db()
    acumuladas += len(nuevas)
    print(f"  [polling {ciclo}] +{len(nuevas)} fila(s) nueva(s) · acumulado {acumuladas}")
print(f"\n[tiempo real] el pipeline capturó {acumuladas} compras nuevas durante la demo")

## 5 · TRANSFORM — limpieza del dataset
Cada operación imprime su efecto (antes → después).

In [ ]:
# Releemos TODO el dataset de hidratación ya con lo insertado por el poller
df = pd.read_sql(SQL_HIDRATACION, conn)
print(f"[limpieza 0] dataset crudo: {df.shape[0]} filas x {df.shape[1]} cols")
print(df.dtypes.to_string(), "\n")

# 1) Duplicados exactos
antes = len(df)
df = df.drop_duplicates()
print(f"[limpieza 1] duplicados: {antes} -> {len(df)} filas ({antes - len(df)} eliminadas)")

# 2) Nulos por columna (y política: referencia_pago puede ser nula, el resto no)
print("[limpieza 2] nulos por columna:")
print(df.isna().sum().to_string())
df = df.dropna(subset=["created_at", "producto", "cantidad", "precio_unit"])
print(f"    tras exigir campos clave: {len(df)} filas")

# 3) Tipos correctos (numeric de Postgres llega como object/Decimal)
df["precio_unit"] = df["precio_unit"].astype(float)
df["cantidad"] = df["cantidad"].astype(int)
df["importe"] = df["precio_unit"] * df["cantidad"]
print(f"[limpieza 3] tipos: precio_unit={df.precio_unit.dtype}, cantidad={df.cantidad.dtype}, importe calculado")

# 4) Zona horaria: UTC -> America/Merida (la hora LOCAL es la que importa para el traslado)
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["ts_local"] = df["created_at"].dt.tz_convert("America/Merida")
df["fecha"] = df["ts_local"].dt.date
df["hora"] = df["ts_local"].dt.hour
print(f"[limpieza 4] tz convertida: {df['created_at'].iloc[-1]} UTC -> {df['ts_local'].iloc[-1]} Mérida")

# 5) Normalización de texto (minúsculas y sin acentos, para categorizar estable)
def normalizar(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")
df["producto_norm"] = df["producto"].map(normalizar)
df["categoria"] = df["producto_norm"].str.extract(r"(agua|electrolit|suero|hidrat)", expand=False).fillna("otra")
print(f"[limpieza 5] categorías detectadas: {df['categoria'].value_counts().to_dict()}")

print(f"\n[limpieza ✓] dataset final: {len(df)} filas limpias")
df[["ts_local", "producto", "categoria", "cantidad", "importe", "hora"]].tail(5)

## 6 · Indicadores ligados a la pregunta de investigación
**IHM — Índice de Hidratación Matutina post-traslado:** proporción de las compras de hidratación que ocurren en la ventana 07:00–10:59 (hora local), justo después del traslado matutino bajo el sol. Complemento: correlación diaria entre compras y UV/temperatura.

In [ ]:
VENTANA = (7, 10)  # 07:00-10:59, llegada del traslado matutino
df["post_traslado"] = df["hora"].between(*VENTANA)

total_unidades = int(df["cantidad"].sum())
unidades_ventana = int(df.loc[df.post_traslado, "cantidad"].sum())
ihm = unidades_ventana / total_unidades * 100 if total_unidades else 0.0

print("="*58)
print(f"  IHM (Índice de Hidratación Matutina post-traslado)")
print(f"  {unidades_ventana} de {total_unidades} unidades en la ventana 07-11h -> IHM = {ihm:.1f}%")
print("="*58)

print("\nUnidades de hidratación por hora local:")
por_hora = df.groupby("hora")["cantidad"].sum()
print(por_hora.to_string())

# Correlación diaria con el clima (días con datos en ambas fuentes)
por_dia = df.groupby("fecha")["cantidad"].sum().rename("unidades").reset_index()
cruce = por_dia.merge(df_clima, on="fecha", how="inner")
print(f"\nDías con compras Y clima: {len(cruce)}")
if len(cruce) >= 3:
    print(f"  corr(unidades, uv_max)   = {cruce.unidades.corr(cruce.uv_max):+.3f}")
    print(f"  corr(unidades, temp_max) = {cruce.unidades.corr(cruce.temp_max):+.3f}")
else:
    print("  ⚠ Aún hay pocos días con traslape para una correlación estable;")
    print("    el poller seguirá acumulando datos durante la semana (checkpoint).")
cruce.tail(5)

## 7 · ETL visual
Diagrama del pipeline con los volúmenes reales de esta corrida y gráficas del indicador.

In [ ]:
# --- 7a: diagrama del pipeline (con conteos reales) ---
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axis("off")

def caja(x, y, w, h, titulo, detalle, color=MORADO):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color, edgecolor="none", zorder=2))
    ax.text(x + w/2, y + h*0.62, titulo, ha="center", va="center",
            color="white", fontsize=10.5, fontweight="bold", zorder=3)
    ax.text(x + w/2, y + h*0.30, detalle, ha="center", va="center",
            color="white", fontsize=8.5, zorder=3)

def flecha(x0, x1, y):
    ax.annotate("", xy=(x1, y), xytext=(x0, y),
                arrowprops=dict(arrowstyle="-|>", lw=2, color="#6b7280"))

caja(0.00, 0.68, 0.235, 0.27, "EXTRACT · Postgres", f"{len(df_db)} compras en vivo")
caja(0.00, 0.37, 0.235, 0.27, "EXTRACT · CSV repo", f"{len(df_prod_hist)} productos hist.")
caja(0.00, 0.06, 0.235, 0.27, "EXTRACT · Open-Meteo", f"{len(df_clima)} días de clima")
caja(0.38, 0.37, 0.24, 0.27, "TRANSFORM", f"{len(df)} filas limpias · 5 ops")
caja(0.76, 0.37, 0.24, 0.27, "INDICADOR", f"IHM = {ihm:.1f}%", color="#2e1355")

# Conectores: las 3 fuentes convergen en TRANSFORM y de ahí al INDICADOR.
flecha(0.245, 0.375, 0.505)
flecha(0.625, 0.755, 0.505)
ax.plot([0.245, 0.31, 0.31, 0.375], [0.815, 0.815, 0.505, 0.505], color="#6b7280", lw=2)
ax.plot([0.245, 0.31, 0.31, 0.375], [0.195, 0.195, 0.505, 0.505], color="#6b7280", lw=2)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Pipeline ETL — Commute & Well-Being (avance)", fontsize=12, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# --- 7b: compras de hidratación por hora, con la ventana post-traslado ---
horas = range(6, 21)
valores = [int(por_hora.get(h, 0)) for h in horas]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axvspan(VENTANA[0] - 0.5, VENTANA[1] + 0.5, color=DORADO, alpha=0.18, zorder=1)
barras = ax.bar(list(horas), valores, color=MORADO, width=0.72, zorder=2)
if max(valores) > 0:
    imax = valores.index(max(valores))
    ax.text(list(horas)[imax], max(valores) + 0.15, str(max(valores)),
            ha="center", fontsize=9, fontweight="bold", color="#1f2937")
ax.text(sum(VENTANA)/2, ax.get_ylim()[1]*0.95, "ventana post-traslado",
        ha="center", fontsize=8.5, color="#7a5b13")
ax.set_xlabel("Hora local (Mérida)"); ax.set_ylabel("Unidades de hidratación")
ax.set_title("¿Cuándo se hidrata el campus? Compras por hora", loc="left")
ax.set_xticks(list(horas)); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
print(f"Lectura: {ihm:.1f}% de las unidades se compran en la ventana 07-11h (IHM).")

In [ ]:
# --- 7c: clima vs compras por día (dos paneles, mismo eje X — sin doble eje) ---
if len(cruce) >= 2:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True,
                                 gridspec_kw={"height_ratios": [1, 1]})
    a1.plot(cruce.fecha, cruce.uv_max, color=MORADO, marker="o", lw=2, label="UV máx")
    a1.plot(cruce.fecha, cruce.temp_max, color=DORADO, marker="s", lw=2, label="Temp máx (°C)")
    a1.legend(frameon=False, fontsize=9); a1.set_ylabel("Exposición")
    a1.spines[["top", "right"]].set_visible(False)
    a1.set_title("Sol/calor del día vs compras de hidratación", loc="left")
    a2.bar(cruce.fecha, cruce.unidades, color=MORADO, width=0.6)
    a2.set_ylabel("Unidades"); a2.set_xlabel("Fecha")
    a2.spines[["top", "right"]].set_visible(False)
    fig.autofmt_xdate()
    plt.tight_layout(); plt.show()
else:
    print("Aún no hay suficientes días con traslape compras+clima para esta gráfica")
    print("(el poller acumulará más días — este avance es un checkpoint).")

## 8 · Estado del avance y próximos pasos

**Hecho en este checkpoint**
- 3 conexiones con logs visibles (Postgres de la plataforma, CSV del repo, API de clima).
- Poller + polling funcionando: el pipeline captura compras nuevas en vivo.
- 5 operaciones de limpieza documentadas con su efecto impreso.
- Indicador IHM calculado y visualizado; primer cruce diario con UV/temperatura.

**Limitaciones honestas**
- Pocos días de traslape compras↔clima todavía: la correlación aún no es estable.
- Las compras de hidratación son un *proxy* de exposición al sol, no una medida directa de estrés/burnout/sueño.

**Próximos pasos**
1. Dejar el poller acumulando varios días (ya corre también como script: `etl/poller_agora.py`).
2. Encuesta corta de estrés percibido/calidad de sueño para cruzar con el IHM por persona.
3. Cargar el indicador diario a una tabla `etl_indicadores` (fase LOAD completa) y automatizarlo.

---
*Fin del cierre de sesión: la conexión a la BD se libera abajo.*

In [ ]:
cur.close()
conn.close()
print("[cierre] conexión a la plataforma liberada. Fin del avance.")